In [1]:
# 2026-03-09 15:26

import os
from google import genai
from PIL import Image

# Initialize the client (it will automatically pick up your GEMINI_API_KEY environment variable)
client = genai.Client()

# Define the paths to your local images. Update these to match your file system.
image_paths = [
    'am105.jpg',
    'am106.jpg',
    'am107.jpg'
]

# Load the images
try:
    images = [Image.open(path) for path in image_paths]
except FileNotFoundError as e:
    print(f"Error loading images: {e}. Please check your file paths.")
    images = []

if images:
    # Define the specific instructions for the model
    prompt_instructions = (
        "Analyze these images and generate a highly detailed written description "
        "of the woman's outfit, suitable for use as a prompt for an AI image generator. "
        "You must include specific, granular details regarding her pose, garments, "
        "accessories, hairstyle, makeup, and shoes."
    )

    # The 'contents' argument accepts a list combining both the images and the text prompt
    request_contents = images + [prompt_instructions]

    # Generate the response using the multimodal flash model
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=request_contents
    )

    # Display the resulting prompt
    print(response.text)

A slender, light-skinned woman with a fair complexion stands gracefully against a crisp, seamless white studio background, illuminated by bright, even lighting. Her light ash-blonde hair is neatly pulled back into a sleek, low bun or chignon, revealing her elegant neck and a minimalist profile.

Her makeup is natural and refined, featuring subtly defined eyebrows, soft brown eyes with a gentle gaze, and full lips adorned with a nude-pink lipstick. Her skin appears flawless and radiant, with a subtle luminous finish.

She is wearing a classic and highly tailored black sheath dress that falls just above her knees. The dress features a modest, round crew neckline and short, cap-style sleeves that slightly extend past her shoulders. The defining characteristic of the dress is its precise tailoring, highlighted by prominent vertical princess seams that run from the bust down through the bodice, and additional vertical seams visible along the sides of the skirt, creating a figure-flattering,

In [4]:
# 2026-03-09 15:47

from google import genai
from google.genai import types
from PIL import Image
from IPython.display import display

# Initialize the client (uses your GEMINI_API_KEY environment variable)
client = genai.Client()

# Load the reference image
reference_image_path = 'z17.jpg'
try:
    reference_image = Image.open(reference_image_path)
except FileNotFoundError:
    print(f"Error: Could not find '{reference_image_path}'.")
    reference_image = None

# Assuming 'detailed_outfit_prompt' is the string we generated in the previous step
# If running this independently, define it here:
detailed_outfit_prompt = "Please generate a ultraphotorealistic image of Zoe (see attached image z17.jpg). She is 160 cm tall, her vital statistics are Bust 37 inches, Waist 28 inches, Hips 39 inches and she has a Bottom Hourglass figure. She is standing in her office at Imperial College where she is a Lecturer in Theoretical Physics. Zoe is a light-skinned woman with a fair complexion stands gracefully. Her red hair is neatly pulled back into a sleek, low bun or chignon, revealing her elegant neck and a minimalist profile. Her makeup is natural and refined, featuring subtly defined eyebrows, soft brown eyes with a gentle gaze, and full lips adorned with a pale pink lipstick. Her skin appears flawless and radiant, with a subtle luminous finish. She is wearing a classic and highly tailored black sheath dress that falls just above her knees. The dress features a modest, round crew neckline and short, cap-style sleeves that slightly extend past her shoulders. The defining characteristic of the dress is its precise tailoring, highlighted by prominent vertical princess seams that run from the bust down through the bodice, and additional vertical seams visible along the sides of the skirt, creating a figure-flattering, body-skimming silhouette. The fabric has a matte, structured appearance, suggesting a high-quality crepe or Ponte knit, giving it a sophisticated and professional finish. On her feet, she wears chic, pointed-toe pumps with a medium-height heel. The shoes feature a distinctive and eye-catching snakeskin or python print in a sophisticated palette of beige, light brown, grey, and subtle black tones, adding a touch of edgy elegance to the classic black dress." 

if reference_image and 'detailed_outfit_prompt' in locals() and detailed_outfit_prompt:
    print("Generating new image with Gemini 3.1 Flash Image Preview (Nano Banana 2)...")
    
    # Construct the edit prompt based on the "High-fidelity detail preservation" best practices
    edit_prompt = (
        f"Using the provided image, change the subject's outfit to match the following description: "
        f"'{detailed_outfit_prompt}'. Ensure that the features of the person's face, her pose, "
        f"and her overall identity remain completely unchanged. Keep the original background and lighting."
    )
    
    # Call the Nano Banana 2 preview model
    response = client.models.generate_content(
        model="gemini-2.5-flash-image",
        contents=[edit_prompt, reference_image],
        config=types.GenerateContentConfig(
            response_modalities=["IMAGE"], # We only want the final image back
            image_config=types.ImageConfig(
                aspect_ratio="3:4", # Good default for portraits, adjust as needed
                image_size="1K"     # Generates a 1024px image
            )
        )
    )
    
    # Process the response, skipping "thought" images to display the final result
    image_found = False
    for part in response.parts:
        # The model uses a "Thinking" process and may return interim images. 
        # We skip those to get to the final rendered output.
        if part.thought: 
            continue
            
        if part.inline_data is not None:
            final_image = part.as_image()
            print("Generation complete! Final Image:")
            display(final_image) # Displays natively in Jupyter
            final_image.save("z17_new_outfit.png")
            print("Saved as 'z17_new_outfit.png'")
            image_found = True
            break
            
    if not image_found:
        print("No final image was returned. The model may have blocked the request due to safety filters.")
else:
    print("Please ensure the reference image is loaded and 'detailed_outfit_prompt' is defined.")

Generating new image with Gemini 3.1 Flash Image Preview (Nano Banana 2)...


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-flash-preview-image\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-flash-preview-image\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-flash-preview-image\nPlease retry in 24.790673736s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-ap